In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

plt.rcParams["figure.dpi"] = 150
plt.style.use("ggplot")


# Load data in a DataFrame
data = pd.read_excel("./Dataset1_BankClients.xlsx")
# Drop the column by its actual name (e.g., 'ID' or the actual name of the column)
data = data.drop(columns=["ID"])  # Replace 'ID' with the actual column name to drop
data.head()

In [ ]:
data.info()

## Check data Quality


### Missing Value


In [ ]:
missing_counts = data.isnull().sum()
missing_percentage = (data.isnull().sum() / len(data)) * 100
missing_info = pd.DataFrame(
    {"Missing Values": missing_counts, "Percentage (%)": missing_percentage}
)
missing_info[missing_info["Missing Values"] > 0].sort_values(
    by="Percentage (%)", ascending=False
)

No missing value observed.


### Data distribution


In [ ]:
categorical_cols = [
    "Gender",
    "Job",
    "Area",
    "CitySize",
    "Investments",
    "FamilySize",
]
numeric_cols = [
    "Age",
    "Income",
    "Wealth",
    "Debt",
    "FinEdu",
    "ESG",
    "Digital",
    "BankFriend",
    "LifeStyle",
    "Luxury",
    "Saving",
]


plt.figure(figsize=(15, 8))
plt.suptitle("Categorical Features Distribution", fontsize=18, y=1.02)

for i, col in enumerate(categorical_cols, 1):
    plt.subplot(2, 3, i)
    sns.countplot(data=data, x=col, hue=col, palette="husl", legend=False)
    plt.title(f"{col}", fontsize=14)
    plt.ylabel("Count")
    plt.xlabel("")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(18, 15))
plt.suptitle("Numeric & Percentile Features Distribution", fontsize=18, y=1.02)

for i, col in enumerate(numeric_cols, 1):
    plt.subplot(4, 3, i)
    sns.histplot(
        data=data, x=col, bins=30, color="#00BFC4", edgecolor="white", alpha=0.8
    )
    plt.title(f"{col}", fontsize=14)
    plt.ylabel("Frequency")
    plt.xlabel("")

plt.tight_layout()
plt.show()

## Demographic & Geographic EDA


In [ ]:
# Feature 4 (Area): 1=Nord, 2=Centro, 3=Sud/Isole
# Feature 7 (Income), Feature 8 (Wealth)
# Create cross-tabulation table with mean values
area_mapping = {1: "Nord", 2: "Centro", 3: "Sud/Isole"}
cross_tab = data.groupby("Area")[["Income", "Wealth"]].mean()
cross_tab.index = cross_tab.index.map(area_mapping)

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "Regional Distribution: Income and Wealth Disparities in Italy",
    fontsize=16,
    fontweight="bold",
    y=1.02,
)

# Prepare data for plotting
regions = list(cross_tab.index)
x_pos = range(len(regions))

# Plot 1: Grouped bar chart
bar_width = 0.35
axes[0].bar(
    [x - bar_width / 2 for x in x_pos],
    cross_tab["Income"],
    bar_width,
    label="Income",
    color="#2E86AB",
    alpha=0.8,
)
axes[0].bar(
    [x + bar_width / 2 for x in x_pos],
    cross_tab["Wealth"],
    bar_width,
    label="Wealth",
    color="#A23B72",
    alpha=0.8,
)
axes[0].set_xlabel("Region", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Mean Value", fontsize=12, fontweight="bold")
axes[0].set_title("Mean Income and Wealth by Region", fontsize=13)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(regions)
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# Add value labels on bars
for i, (inc, wlt) in enumerate(zip(cross_tab["Income"], cross_tab["Wealth"])):
    axes[0].text(
        i - bar_width / 2,
        inc,
        f"{inc:.1f}",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
    )
    axes[0].text(
        i + bar_width / 2,
        wlt,
        f"{wlt:.1f}",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
    )

# Plot 2: Table visualization
axes[1].axis("tight")
axes[1].axis("off")

table_data = []
for region in cross_tab.index:
    table_data.append(
        [
            region,
            f"{cross_tab.loc[region, 'Income']:.2f}",
            f"{cross_tab.loc[region, 'Wealth']:.2f}",
        ]
    )

table = axes[1].table(
    cellText=table_data,
    colLabels=["Region", "Mean Income", "Mean Wealth"],
    cellLoc="center",
    loc="center",
    colColours=["#E8E8E8"] * 3,
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)

# Style the header
for i in range(3):
    table[(0, i)].set_facecolor("#4A90E2")
    table[(0, i)].set_text_props(weight="bold", color="white")

# Color rows based on region
row_colors = ["#90EE90", "#FFD700", "#FFA07A"]  # Nord=green, Centro=yellow, Sud=orange
for i, color in enumerate(row_colors, 1):
    for j in range(3):
        table[(i, j)].set_facecolor(color)
        table[(i, j)].set_alpha(0.3)

axes[1].set_title("Summary Table", fontsize=13, pad=20)

plt.tight_layout()
plt.show()

# Calculate percentage differences from Nord (baseline)
print("\nPercentage Difference from Nord (baseline):")
print("=" * 55)
nord_income = cross_tab.loc["Nord", "Income"]
nord_wealth = cross_tab.loc["Nord", "Wealth"]

for region in cross_tab.index:
    income_diff = ((cross_tab.loc[region, "Income"] - nord_income) / nord_income) * 100
    wealth_diff = ((cross_tab.loc[region, "Wealth"] - nord_wealth) / nord_wealth) * 100
    print(f"{region:12s} | Income: {income_diff:+6.2f}% | Wealth: {wealth_diff:+6.2f}%")

### Job and Investment Preferences Analysis


In [ ]:
# Job type mapping
job_mapping = {
    1: "Unemployed",
    2: "Employee/Worker",
    3: "Manager/Executive",
    4: "Entrepreneur/Freelancer",
    5: "Retired",
}

# Investment type mapping
investment_mapping = {
    1: "No investments",
    2: "Mostly lump sum",
    3: "Mostly capital accumulation",
}

# Create cross-tabulation of Job and Investments
job_investment_crosstab = pd.crosstab(data["Job"], data["Investments"])
job_investment_crosstab.columns = job_investment_crosstab.columns.map(
    investment_mapping
)
job_investment_crosstab.index = job_investment_crosstab.index.map(job_mapping)

# Calculate percentage within each job category
job_investment_pct = (
    pd.crosstab(data["Job"], data["Investments"], normalize="index") * 100
)
job_investment_pct.columns = job_investment_pct.columns.map(investment_mapping)
job_investment_pct.index = job_investment_pct.index.map(job_mapping)

print("Cross-Tabulation: Job Type vs Investment Preferences (Counts)")
print("=" * 70)
print(job_investment_crosstab)
print("\n")
print("Cross-Tabulation: Job Type vs Investment Preferences (Percentage)")
print("=" * 70)
print(job_investment_pct.round(2))
print("\n")

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    "Job Type and Investment Preferences Analysis",
    fontsize=16,
    fontweight="bold",
    y=1.02,
)

# Define color palette for investment types
colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#FFA07A", "#98D8C8", "#F7DC6F"]
investment_types = job_investment_pct.columns.tolist()

# Plot 1: Stacked bar chart (percentages)
job_investment_pct.plot(
    kind="bar",
    stacked=True,
    ax=axes[0],
    color=colors[: len(investment_types)],
    width=0.7,
    edgecolor="white",
    linewidth=1.5,
)
axes[0].set_title(
    "Investment Preferences by Job Type (Percentage)", fontsize=13, fontweight="bold"
)
axes[0].set_xlabel("Job Type", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Percentage (%)", fontsize=12, fontweight="bold")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha="right")
axes[0].legend(title="Investment Type", bbox_to_anchor=(1.05, 1), loc="upper left")
axes[0].grid(axis="y", alpha=0.3)
axes[0].set_ylim(0, 100)

# Add percentage labels on each segment (only if segment > 5%)
for container in axes[0].containers:
    labels = [f"{v:.1f}%" if v > 5 else "" for v in container.datavalues]
    axes[0].bar_label(
        container, labels=labels, label_type="center", fontsize=8, fontweight="bold"
    )

# Plot 2: Grouped bar chart (absolute counts) for better comparison
x_pos = range(len(job_investment_crosstab.index))
bar_width = 0.15
job_labels = job_investment_crosstab.index.tolist()

for i, inv_type in enumerate(investment_types):
    offset = (i - len(investment_types) / 2) * bar_width + bar_width / 2
    axes[1].bar(
        [x + offset for x in x_pos],
        job_investment_crosstab[inv_type],
        bar_width,
        label=inv_type,
        color=colors[i],
        alpha=0.8,
        edgecolor="white",
    )

axes[1].set_title(
    "Investment Preferences by Job Type (Counts)", fontsize=13, fontweight="bold"
)
axes[1].set_xlabel("Job Type", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Count", fontsize=12, fontweight="bold")
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(job_labels, rotation=45, ha="right")
axes[1].legend(title="Investment Type", bbox_to_anchor=(1.05, 1), loc="upper left")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# Statistical analysis - find the most popular investment for each job
print("\nMost Popular Investment Type by Job:")
print("=" * 70)
for job in job_investment_pct.index:
    most_popular = job_investment_pct.loc[job].idxmax()
    percentage = job_investment_pct.loc[job, most_popular]
    print(f"{str(job):20s} -> {str(most_popular):30s} ({percentage:.1f}%)")

# Compare specific job categories (if managers and unemployed exist)
print("\n" + "=" * 70)
print("Comparative Analysis:")
print("=" * 70)

# Show investment distribution for each job type
for job in job_investment_pct.index:
    print(f"\n{job}:")
    for inv_type in investment_types:
        count = job_investment_crosstab.loc[job, inv_type]
        pct = job_investment_pct.loc[job, inv_type]
        print(f"  {str(inv_type):30s}: {count:4d} clients ({pct:5.1f}%)")

# Age and Family Members Distribution Analysis


In [ ]:
# Define life stage categories based on age and family size
def categorize_life_stage(row):
    age = row["Age"]
    family_size = row["FamilySize"]

    if age < 30 and family_size <= 2:
        return "Single Youth"
    elif age < 30 and family_size > 2:
        return "Young Family"
    elif 30 <= age < 50 and family_size <= 2:
        return "Young Couple/Single"
    elif 30 <= age < 50 and family_size > 2:
        return "Family Maturity"
    elif age >= 50 and family_size <= 2:
        return "Empty Nest/Senior"
    else:
        return "Mature Family"


data["LifeStage"] = data.apply(categorize_life_stage, axis=1)

print("Customer Distribution by Life Stage:")
print("=" * 70)
life_stage_counts = data["LifeStage"].value_counts()
life_stage_pct = (data["LifeStage"].value_counts(normalize=True) * 100).round(2)
life_stage_summary = pd.DataFrame(
    {"Count": life_stage_counts, "Percentage (%)": life_stage_pct}
)
print(life_stage_summary)
print("\n")

# Create comprehensive visualization
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

fig.suptitle(
    "Age and Family Size Distribution: Customer Life Stage Analysis",
    fontsize=18,
    fontweight="bold",
    y=0.98,
)

# Plot 1: Age distribution histogram
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(data["Age"], bins=30, color="#3498db", edgecolor="white", alpha=0.8)
ax1.set_xlabel("Age", fontsize=11, fontweight="bold")
ax1.set_ylabel("Frequency", fontsize=11, fontweight="bold")
ax1.set_title("Age Distribution", fontsize=12, fontweight="bold")
ax1.axvline(
    data["Age"].mean(),
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {data['Age'].mean():.1f}",
)
ax1.axvline(
    data["Age"].median(),
    color="orange",
    linestyle="--",
    linewidth=2,
    label=f"Median: {data['Age'].median():.1f}",
)
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

# Plot 2: Family Size distribution
ax2 = fig.add_subplot(gs[0, 1])
family_counts = data["FamilySize"].value_counts().sort_index()
ax2.bar(
    family_counts.index,
    family_counts.values,
    color="#e74c3c",
    edgecolor="white",
    alpha=0.8,
)
ax2.set_xlabel("Family Size", fontsize=11, fontweight="bold")
ax2.set_ylabel("Frequency", fontsize=11, fontweight="bold")
ax2.set_title("Family Size Distribution", fontsize=12, fontweight="bold")
ax2.axvline(
    data["FamilySize"].mean(),
    color="darkred",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {data['FamilySize'].mean():.1f}",
)
ax2.legend()
ax2.grid(axis="y", alpha=0.3)

# Plot 3: Life Stage pie chart
ax3 = fig.add_subplot(gs[0, 2])
colors_pie = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#FFA07A", "#98D8C8", "#F7DC6F"]
wedges, texts, autotexts = ax3.pie(
    life_stage_counts,
    labels=life_stage_counts.index,
    autopct="%1.1f%%",
    startangle=90,
    colors=colors_pie[: len(life_stage_counts)],
)
ax3.set_title("Life Stage Distribution", fontsize=12, fontweight="bold")
for autotext in autotexts:
    autotext.set_color("white")
    autotext.set_fontweight("bold")
    autotext.set_fontsize(9)

# Plot 4: 2D Histogram (Heatmap) - Age vs Family Size
ax4 = fig.add_subplot(gs[1, :])
hist_data = data[["Age", "FamilySize"]].copy()
ax4.hist2d(
    hist_data["Age"],
    hist_data["FamilySize"],
    bins=[30, len(data["FamilySize"].unique())],
    cmap="YlOrRd",
    edgecolors="white",
)
ax4.set_xlabel("Age", fontsize=12, fontweight="bold")
ax4.set_ylabel("Family Size", fontsize=12, fontweight="bold")
ax4.set_title(
    "2D Distribution: Age vs Family Size (Heatmap)", fontsize=13, fontweight="bold"
)
cbar = plt.colorbar(ax4.collections[0], ax=ax4)
cbar.set_label("Frequency", fontsize=10, fontweight="bold")
ax4.grid(alpha=0.3)

# Plot 5: Box plot - Age by Family Size
ax5 = fig.add_subplot(gs[2, 0])
family_sizes = sorted(data["FamilySize"].unique())
box_data = [data[data["FamilySize"] == fs]["Age"].values for fs in family_sizes]
bp = ax5.boxplot(box_data, tick_labels=family_sizes, patch_artist=True)
for patch, color in zip(bp["boxes"], colors_pie):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax5.set_xlabel("Family Size", fontsize=11, fontweight="bold")
ax5.set_ylabel("Age", fontsize=11, fontweight="bold")
ax5.set_title("Age Distribution by Family Size", fontsize=12, fontweight="bold")
ax5.grid(axis="y", alpha=0.3)

# Plot 6: Scatter plot with life stage coloring
ax6 = fig.add_subplot(gs[2, 1])
life_stage_colors = {
    "Single Youth": "#FF6B6B",
    "Young Family": "#4ECDC4",
    "Young Couple/Single": "#45B7D1",
    "Family Maturity": "#FFA07A",
    "Empty Nest/Senior": "#98D8C8",
    "Mature Family": "#F7DC6F",
}
for stage, color in life_stage_colors.items():
    mask = data["LifeStage"] == stage
    ax6.scatter(
        data[mask]["Age"],
        data[mask]["FamilySize"],
        label=stage,
        color=color,
        alpha=0.6,
        s=30,
        edgecolors="white",
        linewidth=0.5,
    )
ax6.set_xlabel("Age", fontsize=11, fontweight="bold")
ax6.set_ylabel("Family Size", fontsize=11, fontweight="bold")
ax6.set_title("Customer Segmentation by Life Stage", fontsize=12, fontweight="bold")
ax6.legend(loc="upper left", fontsize=8, framealpha=0.9)
ax6.grid(alpha=0.3)

# Plot 7: Stacked bar chart - Family Size by Age groups
ax7 = fig.add_subplot(gs[2, 2])
data["AgeGroup"] = pd.cut(
    data["Age"],
    bins=[0, 30, 40, 50, 60, 100],
    labels=["<30", "30-40", "40-50", "50-60", "60+"],
)
age_family_crosstab = (
    pd.crosstab(data["AgeGroup"], data["FamilySize"], normalize="index") * 100
)
age_family_crosstab.plot(
    kind="bar",
    stacked=True,
    ax=ax7,
    color=colors_pie[: len(age_family_crosstab.columns)],
    edgecolor="white",
    linewidth=1,
)
ax7.set_xlabel("Age Group", fontsize=11, fontweight="bold")
ax7.set_ylabel("Percentage (%)", fontsize=11, fontweight="bold")
ax7.set_title("Family Size Distribution by Age Group", fontsize=12, fontweight="bold")
ax7.set_xticklabels(ax7.get_xticklabels(), rotation=45, ha="right")
ax7.legend(title="Family Size", bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
ax7.set_ylim(0, 100)
ax7.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# Statistical summary
print("\nStatistical Summary:")
print("=" * 70)
print(f"Total Customers: {len(data)}")
print("\nAge Statistics:")
print(f"  Mean: {data['Age'].mean():.2f} years")
print(f"  Median: {data['Age'].median():.2f} years")
print(f"  Std Dev: {data['Age'].std():.2f} years")
print(f"  Range: {data['Age'].min()}-{data['Age'].max()} years")
print("\nFamily Size Statistics:")
print(f"  Mean: {data['FamilySize'].mean():.2f} members")
print(f"  Median: {data['FamilySize'].median():.2f} members")
print(f"  Mode: {data['FamilySize'].mode()[0]} members")
print(f"  Range: {data['FamilySize'].min()}-{data['FamilySize'].max()} members")

# Identify main customer groups
print("\n" + "=" * 70)
print("MAIN CUSTOMER GROUP ANALYSIS:")
print("=" * 70)

single_youth = len(data[(data["Age"] < 30) & (data["FamilySize"] <= 2)])
family_maturity = len(
    data[(data["Age"] >= 30) & (data["Age"] < 50) & (data["FamilySize"] > 2)]
)

single_youth_pct = (single_youth / len(data)) * 100
family_maturity_pct = (family_maturity / len(data)) * 100

print(
    f"\nSingle Youth Phase (Age < 30, Family ≤ 2): {single_youth} customers ({single_youth_pct:.1f}%)"
)
print(
    f"Family Maturity Phase (Age 30-50, Family > 2): {family_maturity} customers ({family_maturity_pct:.1f}%)"
)


Single Youth Phase (Age < 30, Family ≤ 2): 273 customers (5.5%)
Family Maturity Phase (Age 30-50, Family > 2): 475 customers (9.5%)

Main customer group: FAMILY MATURITY PHASE
The bank's primary customers are middle-aged (30-50) with larger families,
representing 9.5% of the total customer base.


## Analysis of Differences in Core Behavioral Features

### Bivariate Analysis: Financial Psychology & Behavior Indicators (Features 7–16) vs Investment Type (Feature 17)

This section dives into how customers' psychological and behavioral scores differ across the three investment types:

- **Type 1** – No investments
- **Type 2** – Mostly lump-sum investments
- **Type 3** – Mostly capital accumulation (regular / systematic investing)


In [ ]:
import matplotlib.patches as mpatches
import numpy as np

# ─── Labels & palette ────────────────────────────────────────────────────────
investment_labels = {
    1: "Type 1\n(No Investment)",
    2: "Type 2\n(Lump Sum)",
    3: "Type 3\n(Capital Acc.)",
}
inv_colors = ["#FF6B6B", "#4ECDC4", "#45B7D1"]

behavior_features = [
    "FinEdu",
    "ESG",
    "Digital",
    "Saving",
    "BankFriend",
    "LifeStyle",
    "Luxury",
]
feature_titles = {
    "FinEdu": "Financial Education",
    "ESG": "ESG Awareness",
    "Digital": "Digital Propensity",
    "Saving": "Saving Propensity",
    "BankFriend": "Bank Friendliness",
    "LifeStyle": "Lifestyle Index",
    "Luxury": "Luxury Index",
}

# ─── Multiple Boxplots ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
fig.suptitle(
    "Financial Psychology & Behavior Features by Investment Type\n"
    "(Bivariate Analysis: Behavioral Features vs Target)",
    fontsize=16,
    fontweight="bold",
    y=1.01,
)
axes_flat = axes.flatten()

for idx, feature in enumerate(behavior_features):
    ax = axes_flat[idx]
    plot_data = [data[data["Investments"] == t][feature].values for t in [1, 2, 3]]
    labels = [investment_labels[t] for t in [1, 2, 3]]

    bp = ax.boxplot(
        plot_data,
        tick_labels=labels,
        patch_artist=True,
        medianprops=dict(color="black", linewidth=2.5),
        whiskerprops=dict(linewidth=1.5),
        capprops=dict(linewidth=2),
        flierprops=dict(marker="o", markerfacecolor="gray", markersize=3, alpha=0.5),
    )
    for patch, color in zip(bp["boxes"], inv_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)

    ax.set_title(feature_titles[feature], fontsize=13, fontweight="bold")
    ax.set_ylabel("Score", fontsize=10)
    ax.grid(axis="y", alpha=0.35)

    # Annotate median values
    for i, vals in enumerate(plot_data):
        median_val = np.median(vals)
        ax.text(
            i + 1,
            median_val,
            f"  {median_val:.2f}",
            ha="left",
            va="center",
            fontsize=8.5,
            fontweight="bold",
            color="#222222",
        )

# Hide unused 8th subplot
axes_flat[-1].set_visible(False)

# Legend
patches = [
    mpatches.Patch(color=c, label=l, alpha=0.75)
    for c, l in zip(
        inv_colors,
        ["Type 1: No Investment", "Type 2: Lump Sum", "Type 3: Capital Acc. (Regular)"],
    )
]
fig.legend(
    handles=patches,
    loc="lower right",
    fontsize=11,
    framealpha=0.9,
    bbox_to_anchor=(0.98, 0.02),
)
plt.tight_layout()
plt.show()

# ─── Summary statistics ──────────────────────────────────────────────────────
print("Median Scores by Investment Type:")
print("=" * 75)
summary = data.groupby("Investments")[behavior_features].median()
summary.index = summary.index.map(
    {1: "Type 1 (No Inv.)", 2: "Type 2 (Lump Sum)", 3: "Type 3 (Cap. Acc.)"}
)
print(summary.round(3))


### Business Insights from Boxplots

| Feature                          | Observation                                                                                                                                                              |
| -------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| **Saving Propensity**            | Type 3 customers have a **significantly higher** saving median — customers who save more systematically are far more likely to adopt regular capital-accumulation plans. |
| **Financial Education (FinEdu)** | Type 3 scores higher, confirming that financially literate clients gravitate toward disciplined, long-horizon investing.                                                 |
| **Digital Propensity**           | Type 2 and Type 3 customers are more digitally engaged — a strong signal for cross-sell via mobile/online channels.                                                      |
| **ESG Awareness**                | Relatively uniform across groups: ESG sensitivity alone does not differentiate investment preference for this portfolio.                                                 |
| **LifeStyle / Luxury**           | Lower for Type 3 — regular investors tend to be more frugal, reinforcing the saving → systematic investment linkage.                                                     |

> **Actionable Takeaway**: Customers in the _No Investment_ group (Type 1) who exhibit above-average Saving scores are the most promising conversion targets for capital-accumulation products. A nudge campaign highlighting the link between saving habits and systematic investing could materially move conversion rates.


In [ ]:
continuous_features = [
    "Age",
    "FamilySize",
    "Income",
    "Wealth",
    "Debt",
    "FinEdu",
    "ESG",
    "Digital",
    "BankFriend",
    "LifeStyle",
    "Luxury",
    "Saving",
]

available_features = [c for c in continuous_features if c in data.columns]
missing_features = [c for c in continuous_features if c not in data.columns]

X = data[available_features].copy()
investments = data["Investments"].copy()

# Pearson Correlation Heatmap
corr_matrix = X.corr(method="pearson")
plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    square=True,
    cbar_kws={"label": "Pearson Correlation"},
)
plt.title(
    "Feature Correlation Heatmap (Continuous Features: Age, FamilySize, Features 7-16)",
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

# Business Insights and Risk Detection (|correlation| > 0.8)
threshold = 0.8
strong_pairs = []
for i, col1 in enumerate(corr_matrix.columns):
    for j, col2 in enumerate(corr_matrix.columns):
        if j <= i:
            continue
        corr_val = corr_matrix.loc[col1, col2]
        if abs(corr_val) > threshold:
            relation = "positive" if corr_val > 0 else "negative"
            strong_pairs.append((col1, col2, corr_val, relation))

strong_pairs = sorted(strong_pairs, key=lambda x: abs(x[2]), reverse=True)

print(f"Strong Correlations Detected (|r| > {threshold}):")
print("=" * 72)
if strong_pairs:
    for col1, col2, corr_val, relation in strong_pairs:
        print(f"  {col1:12s} vs {col2:12s} | r = {corr_val:+.3f} ({relation})")
    print("\nMulticollinearity Note:")
    print(
        "  In downstream regression, consider removing one variable from each strongly"
    )
    print(
        "  correlated pair (or use regularization/VIF checks) to reduce multicollinearity."
    )
else:
    print("  No pairs exceeded the threshold.")

if {"Income", "Wealth"}.issubset(corr_matrix.columns):
    iw_corr = corr_matrix.loc["Income", "Wealth"]
    print(f"\nCheck - Income vs Wealth: r = {iw_corr:+.3f}")

if {"Luxury", "Saving"}.issubset(corr_matrix.columns):
    ls_corr = corr_matrix.loc["Luxury", "Saving"]
    print(f"Check - Luxury vs Saving: r = {ls_corr:+.3f}")

In [ ]:
# Initial 2D reduction before clustering: PCA and t-SNE
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca_2d = PCA(n_components=2, random_state=42)
X_pca = pca_2d.fit_transform(X_scaled)
explained = pca_2d.explained_variance_ratio_
print(f"\nPCA explained variance: PC1={explained[0]:.2%}, PC2={explained[1]:.2%}")

# t-SNE can be expensive on large data; sample for responsiveness if needed.
max_tsne_points = 2000
if len(X_scaled) > max_tsne_points:
    sample_idx = np.random.default_rng(42).choice(
        len(X_scaled), size=max_tsne_points, replace=False
    )
    X_tsne_input = X_scaled[sample_idx]
    investments_tsne = investments.iloc[sample_idx]
    print(
        f"t-SNE computed on a random sample of {max_tsne_points} points out of {len(X_scaled)}."
    )
else:
    X_tsne_input = X_scaled
    investments_tsne = investments

tsne_2d = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=42,
    max_iter=1000,
)
X_tsne = tsne_2d.fit_transform(X_tsne_input)

investment_labels = {
    1: "Type 1: No Investment",
    2: "Type 2: Lump Sum",
    3: "Type 3: Capital Accumulation",
}
plot_palette = {1: "#FF6B6B", 2: "#4ECDC4", 3: "#45B7D1"}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    "2D Projection Before Clustering (Colored by Investment Preference)",
    fontsize=14,
    fontweight="bold",
)

# PCA scatter
for inv_type in sorted(investments.unique()):
    mask = investments == inv_type
    axes[0].scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        s=18,
        alpha=0.65,
        color=plot_palette.get(inv_type, "gray"),
        label=investment_labels.get(inv_type, f"Type {inv_type}"),
    )
axes[0].set_title("PCA 2D")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].grid(alpha=0.25)

# t-SNE scatter
for inv_type in sorted(investments_tsne.unique()):
    mask = investments_tsne == inv_type
    axes[1].scatter(
        X_tsne[mask, 0],
        X_tsne[mask, 1],
        s=18,
        alpha=0.65,
        color=plot_palette.get(inv_type, "gray"),
        label=investment_labels.get(inv_type, f"Type {inv_type}"),
    )
axes[1].set_title("t-SNE 2D")
axes[1].set_xlabel("Dim 1")
axes[1].set_ylabel("Dim 2")
axes[1].grid(alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=3,
    frameon=True,
    bbox_to_anchor=(0.5, 1.02),
)
plt.tight_layout()
plt.show()

With current threshold 0.8, multicollinearity removal is not immediately required from this feature set.

For clustering, separation by Investments appears weak in 2D projection, so expect overlap unless you engineer additional discriminative features or try other embeddings/clustering settings.
